# Project_Cam Academy Edition — one session, end to end

This notebook runs the full **camera-only metric stack** on one synthetic 10-minute
small-sided training block (8 players, 15 FPS — the current rig's measured rate) and
produces the deliverables an academy coach sees: per-player load reports (EN/RU/KK),
an ACWR injury-risk band, and a tactical snapshot.

On the live rig the synthetic trajectories are replaced by the
detection → tracking → pitch-homography stream; **everything downstream of the
trajectory is exactly this code** (`src/project_cam/metrics/`).

Mirrors `demo/run_academy_demo.py` (kept in lockstep; the script is the CI-checkable copy).

In [ ]:
import sys
sys.path.insert(0, "../src")
sys.path.insert(0, ".")

import numpy as np
import matplotlib.pyplot as plt

from run_academy_demo import synth_trajectory, FPS, PITCH_L, PITCH_W, POSITION_SIGMA_M
from project_cam.metrics import (
    acwr, physical_load, render_session_report, team_shape, voronoi_control,
)

rng = np.random.default_rng(7)
players = [f"{i+1:02d}" for i in range(8)]
teams = {p: ("A" if i < 4 else "B") for i, p in enumerate(players)}
trajectories = {p: synth_trajectory(rng) for p in players}
print(f"{len(players)} players, {len(trajectories['01'][1])} frames @ {FPS} FPS")

## 1. Physical load — the GPS-vest replacement
Distance, HSR/sprint zones, accel/decel events, metabolic power — each with a
propagated uncertainty from the tracking noise (`position_sigma_m`).

In [ ]:
pos, t = trajectories["03"]
phys = physical_load(pos, t, position_sigma_m=POSITION_SIGMA_M)
history = list(rng.normal(2800, 400, size=27).clip(min=0))
load = acwr(history + [phys.total_distance_m])
print(render_session_report("03", phys.to_dict(), load.to_dict(), lang="en"))

In [ ]:
speed_kmh = np.linalg.norm(np.diff(pos, axis=0), axis=1) / np.diff(t) * 3.6
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.plot(pos[:, 0], pos[:, 1], lw=0.4)
ax1.set(title="Player 03 trajectory", xlim=(0, PITCH_L), ylim=(0, PITCH_W),
        xlabel="m", ylabel="m", aspect="equal")
ax2.plot(t[:-1] / 60, speed_kmh, lw=0.5)
ax2.axhline(19.8, color="orange", ls="--", label="HSR 19.8 km/h")
ax2.axhline(25.2, color="red", ls="--", label="Sprint 25.2 km/h")
ax2.set(title="Speed profile", xlabel="min", ylabel="km/h")
ax2.legend()
plt.tight_layout()

## 2. Tactical snapshot — pitch control + team shape
Voronoi (nearest-player) occupancy at mid-session; upgradeable to the
velocity-aware Spearman/Shaw model without changing this call.

In [ ]:
mid = len(t) // 2
team_a = np.array([trajectories[p][0][mid] for p in players if teams[p] == "A"])
team_b = np.array([trajectories[p][0][mid] for p in players if teams[p] == "B"])
control = voronoi_control(team_a, team_b, pitch_length=PITCH_L, pitch_width=PITCH_W, grid_step=0.25)

plt.figure(figsize=(8, 4))
plt.imshow(control["grid"], origin="lower", extent=[0, PITCH_L, 0, PITCH_W],
           cmap="coolwarm", alpha=0.35, aspect="equal")
plt.scatter(team_a[:, 0], team_a[:, 1], c="blue", s=80, label="Team A")
plt.scatter(team_b[:, 0], team_b[:, 1], c="red", s=80, label="Team B")
plt.title(f"Pitch control: A {control['team_a']:.0%} / B {control['team_b']:.0%}")
plt.legend()
print(team_shape(team_a))

## 3. Whole-squad session table
The batch path (`demo/run_academy_demo.py`) writes this as CSV + per-player
markdown reports in three languages to `demo/output/`.

In [ ]:
for p in players:
    pp, tt = trajectories[p]
    s = physical_load(pp, tt, position_sigma_m=POSITION_SIGMA_M)
    print(f"#{p} ({teams[p]})  {s.total_distance_m:7.1f} m ±{s.total_distance_sigma_m:.0f}"
          f"  HSR {s.hsr_distance_m:6.1f} m  sprints {s.sprint_count}"
          f"  vmax {s.max_speed_kmh:4.1f} km/h  [{s.confidence}]")

## What replaces the synthetic input on the live rig

| Stage | Source |
|---|---|
| Frames | 4–6 cameras, `src/project_cam/streaming/` + `Parallel_working` capture |
| Detection | detector service (subprocess-isolated, see `STACK.md` §isolation) |
| Tracking | ByteTrack + OSNet ReID (`feat/detect-track-isolated`) |
| Pitch coords | charuco/pitch-keypoint calibration (`feat/auto-calibration-v2`) |
| 3D pose | RTMPose 2D + triangulation (`triangulate_multi`, protected) |
| Metrics | **this package** — identical code path |